<a href="https://colab.research.google.com/github/Dkhan213/Etch-AI-Optimization/blob/main/AI_Optimization_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 4.1 MB/s eta 0:00:00


In [4]:
# ==========================================
# ENVIRONMENT SETUP
# ==========================================
# Quietly install the scikit-optimize machine learning library into Google Colab (-q means quiet)
!pip install scikit-optimize -q

import pandas as pd             # Handles data tables (like reading CSV files)
import numpy as np              # Handles complex math and array calculations
from skopt import Optimizer     # The core AI engine that performs Bayesian Optimization
from skopt.space import Space, Real  # Allows us to define physical variable ranges (continuous numbers)
import warnings

# Hide harmless background warnings from the libraries to keep the output clean
warnings.filterwarnings('ignore')


# ==========================================
# GITHUB DATA SOURCES
# ==========================================
# Direct links to raw GitHub CSV files so Colab can read them from anywhere
HISTORICAL_SOURCE = "https://raw.githubusercontent.com/dkhan213/etch-ai-optimization/main/historical_baseline.csv"
NANOFAB_SOURCE = "https://raw.githubusercontent.com/dkhan213/etch-ai-optimization/main/nanofab_experiments.csv"


# ==========================================
# STEP 1: DATA INGESTION & FEATURE ENGINEERING
# ==========================================
print("SYSTEM CHECK: Loading live data from GitHub...")

# Attempt to load the historical academic data
try:
    external_df = pd.read_csv(HISTORICAL_SOURCE)
except Exception as e:
    # If GitHub is down or the link breaks, create an empty table so the script doesn't crash
    print(f"Notice: Could not load historical baseline ({e}). Using empty DataFrame.")
    external_df = pd.DataFrame(columns=['rf_power', 'sf6_flow', 'o2_flow', 'etch_rate'])

# Attempt to load your actual cleanroom experiments
try:
    nanofab_df = pd.read_csv(NANOFAB_SOURCE)
except Exception:
    nanofab_df = pd.DataFrame(columns=['rf_power', 'sf6_flow', 'o2_flow', 'etch_rate'])

# ACTIVE LEARNING & TRANSFER LEARNING SWITCH:
if len(nanofab_df) >= 3:
    print(f"Nanofab threshold reached ({len(nanofab_df)} runs logged). Training strictly on local UH hardware.")
    active_data = nanofab_df.copy()                           #  Creates an explicit copy of nanofab data to prevent pandas slice warnings
    y_target = (-active_data['etch_rate']).values.tolist()    #  Directly negates raw local etch rates for maximization
else:
    print(f"Insufficient Nanofab data ({len(nanofab_df)} runs logged). Seeding prior with {len(external_df)} records.")
    active_data = external_df.copy()                          #  Creates an explicit copy of external baseline data

    # TARGET NORMALIZATION (MIN-MAX SCALING):
    # Scales literature etch rates between 0.0 and 1.0 so the AI learns the relative physical trends
    # (peaks and valleys) without being confused by high-pressure or high-power absolute numbers.
    if len(active_data) > 0:                                  #  Checks that external data exists before attempting scaling
        min_rate = active_data['etch_rate'].min()             #  Finds the lowest etch rate entry in the external dataset
        max_rate = active_data['etch_rate'].max()             #  Finds the highest etch rate entry in the external dataset
        if max_rate != min_rate:                              #  Prevents division by zero if all historical etch rates happen to be equal
            scaled_rate = (active_data['etch_rate'] - min_rate) / (max_rate - min_rate) # NEW: Applies standard Min-Max formula (range 0.0 to 1.0)
        else:
            scaled_rate = active_data['etch_rate'] * 0 + 1.0   #  Assigns uniform baseline score if max equals min
        y_target = (-scaled_rate).values.tolist()             #  Negates normalized score so skopt can maximize it via minimization
    else:
        y_target = []                                         #  Keeps target empty if no data is loaded

# FEATURE ENGINEERING:
# Computes O2/SF6 ratio so the AI learns chemical passivation balance regardless of total flow scale.
if len(active_data) > 0:                                      #  Ensures active_data is non-empty before feature calculation
    active_data['o2_sf6_ratio'] = active_data['o2_flow'] / active_data['sf6_flow'] #  Derives gas flow ratio column for chemical learning

# Isolate feature matrix X (includes 4 features: RF Power, SF6 Flow, O2 Flow, and O2/SF6 Ratio)
if len(active_data) > 0:                                      #  Checks if active data is available
    X_prior = active_data[['rf_power', 'sf6_flow', 'o2_flow', 'o2_sf6_ratio']].values.tolist() #  Extracts 4D feature set as a list of lists
else:
    X_prior = []                                              #  Defines empty list fallback if no input data exists


# ==========================================
# STEP 2: DYNAMIC SEARCH SPACE & OPTIMIZER
# ==========================================
# DYNAMIC BOUND EXPANSION:
# Automatically stretches the AI's internal search space to fit high-flow or high-power literature papers
# while keeping output candidate selection strictly inside UH cleanroom safety bounds.
if len(X_prior) > 0:                                          #  Evaluates dataset bounds if data is loaded
    rf_min = min(20.0, float(active_data['rf_power'].min()))   #  Dynamically finds minimum RF power in dataset
    rf_max = max(160.0, float(active_data['rf_power'].max()))  #  Expands upper RF power bound if paper uses higher wattage
    sf6_min = min(20.0, float(active_data['sf6_flow'].min()))  #  Dynamically finds minimum SF6 flow in dataset
    sf6_max = max(50.0, float(active_data['sf6_flow'].max()))  #  Expands upper SF6 bound for high-flow papers (e.g. 180+ SCCM)
    o2_min = min(5.0, float(active_data['o2_flow'].min()))     #  Dynamically finds minimum O2 flow in dataset
    o2_max = max(50.0, float(active_data['o2_flow'].max()))    #  Expands upper O2 bound for high-flow papers
    ratio_min = float(active_data['o2_sf6_ratio'].min())       #  Finds lower O2/SF6 ratio bound in dataset
    ratio_max = float(active_data['o2_sf6_ratio'].max())       #  Finds upper O2/SF6 ratio bound in dataset
else:
    rf_min, rf_max = 20.0, 160.0                              #  Fallback default RF power bounds
    sf6_min, sf6_max = 20.0, 200.0                             #  Fallback default SF6 flow bounds accommodating high-flow papers
    o2_min, o2_max = 5.0, 150.0                                #  Fallback default O2 flow bounds accommodating high-flow papers
    ratio_min, ratio_max = 0.1, 1.0                            #  Fallback default chemical ratio bounds

search_space = [
    Real(rf_min, rf_max, name='rf_power'),
    Real(sf6_min, sf6_max, name='sf6_flow'),
    Real(o2_min, o2_max, name='o2_flow'),
    Real(ratio_min, ratio_max, name='o2_sf6_ratio')            #  Adds O2/SF6 ratio as a 4th continuous dimension in AI search space
]

print("\nCALCULATING: Generating optimal parameters...")

# Configure the Bayesian AI Engine
ai_engine = Optimizer(
    dimensions=search_space,  # Tells the AI the 4 variable axes it is allowed to think about
    base_estimator="RF",      # Uses a "Random Forest" model to predict the etch rate landscape
    n_initial_points=0,       # Set to 0 to stop the AI from making random guesses; we want it to use our data immediately
    acq_func="EI",            # Expected Improvement: The math that balances exploring unknown areas vs exploiting known peaks
    random_state=42           # Locks the random seed so the script gives the exact same result if run twice
)

# STRICT UH CLEANROOM SAFETY LIMITS (Oxford System 100 Hardware Window)
safe_limits = Space([
    Real(25.0, 60.0),   # Safe RF Substrate Power [W]
    Real(20.0, 40.0),   # Safe SF6 Gas Flow [SCCM]
    Real(5.0, 15.0)     # Safe O2 Gas Flow [SCCM]
])

# Randomly generate 1,000 potential recipes that strictly obey the UH cleanroom safety limits above
safe_candidates_raw = safe_limits.rvs(1000, random_state=42)

# CANDIDATE FEATURE ALIGNMENT:
# Appends the calculated O2/SF6 ratio to each 3D safe recipe so it matches the 4D input format expected by the AI.
safe_candidates_4d = []                                       #  Initializes list for 4D candidate vectors
for cand in safe_candidates_raw:                              #  Iterates through each 3D safe candidate
    ratio = cand[2] / cand[1]                                 #  Computes candidate O2 flow divided by candidate SF6 flow
    safe_candidates_4d.append([cand[0], cand[1], cand[2], ratio]) #  Appends complete 4D vector [RF, SF6, O2, Ratio]

# TELL: Feed the isolated data (X and Y) into the AI so it learns the physics
if len(X_prior) > 0:
    ai_engine.tell(X_prior, y_target)                         #  Feeds 4D feature matrix and normalized target array into skopt

    # SAFETY OVERRIDE:
    # We force the AI to evaluate all 1,000 safe 4D recipes and predict performance for each.
    predicted_scores = ai_engine.models[-1].predict(safe_candidates_4d) #  Predicts performance scores across 4D candidates

    # Find the specific recipe index that yielded the best predicted performance score
    best_safe_idx = np.argmin(predicted_scores)               #  Identifies candidate index with minimum negative score

    # Lock in that specific safe recipe as our next experiment
    next_experiment = safe_candidates_raw[best_safe_idx]
else:
    # Fallback to the first safe candidate if both GitHub links fail entirely
    next_experiment = safe_candidates_raw[0]


# ==========================================
# STEP 3: OUTPUT PROTOCOL
# ==========================================
print("\n--- TEAM ETCH-A-SKETCH: ML RECIPE FOR NEXT LAB ITERATION ---")
print("1. Set static parameters on the Oxford System 100:")
print("   -> ICP Source Power: 800.0 [W]")
print("   -> Chamber Pressure: 20.0  [mTorr]")
print("   -> Argon (Ar) Flow:  15.0  [SCCM]")
print("   -> Process Time:     120   [Seconds]")
print("\n2. Input the AI-OPTIMIZED variables:")
print(f"   -> RF Substrate Bias: {next_experiment[0]:.1f} [W]")
print(f"   -> SF6 Gas Flow:      {next_experiment[1]:.1f} [SCCM]")
print(f"   -> O2 Gas Flow:       {next_experiment[2]:.1f} [SCCM]")

SYSTEM CHECK: Loading live data from GitHub...
Insufficient Nanofab data (0 runs logged). Seeding prior with 13 records.

CALCULATING: Generating optimal parameters...

--- TEAM ETCH-A-SKETCH: ML RECIPE FOR NEXT LAB ITERATION ---
1. Set static parameters on the Oxford System 100:
   -> ICP Source Power: 800.0 [W]
   -> Chamber Pressure: 20.0  [mTorr]
   -> Argon (Ar) Flow:  15.0  [SCCM]
   -> Process Time:     120   [Seconds]

2. Input the AI-OPTIMIZED variables:
   -> RF Substrate Bias: 58.9 [W]
   -> SF6 Gas Flow:      39.8 [SCCM]
   -> O2 Gas Flow:       5.6 [SCCM]
